In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from db import engine

plt.rcParams["figure.figsize"] = (15, 6)
sns.set_style('whitegrid')

In [2]:
with engine.connect() as conn:
    loc_df = pd.read_sql("select * from location", conn)
    weather_df = pd.read_sql("select * from weather_observation", conn)

In [3]:
df = loc_df.merge(weather_df, left_on='id', right_on='location_id', suffixes=['_loc', '_wea'])

In [4]:
df.isna().sum()

id_loc                      0
city                        0
state                       0
country                     0
latitude                    0
longitude                   0
created_at_loc              0
id_wea                      0
location_id                 0
observed_at                 0
temperature                 0
feels_like                  0
humidity                    0
pressure                    0
wind_speed                  0
weather_description         0
weather_code                0
raw_json               789120
created_at_wea              0
dtype: int64

In [5]:
df['weather_description'].value_counts(dropna=False)

weather_description
Partly cloudy       366469
Clear               358148
Drizzle              22281
Foggy                16296
Thunderstorm         12087
Rain                  7387
Snow                  5611
Rain Showers           698
Freezing Drizzle       106
Freezing Rain           34
Snow Showers             3
Name: count, dtype: int64

> From the above outputs, we see that we have 30 U.S. cities, with hourly weather data ranging from January 1st, 2023 to December 31st, 2025. We have 3 years worth of data, with each city having the same amount of hourly data.


> The only columns in which there is missing data is the raw_json, which we will drop as this is missing for every row, and weather_description. The missing data in weather description means that the data contained a weather code that was not covered in the data dictionary during the transformation process.
>
> NOTE: originally, there was 12087 missing data values. this was fixed in the database directly

In [6]:
df[df['weather_description'].isna()].value_counts('weather_code')

Series([], Name: count, dtype: int64)

Looking at the OpenMeteo documentation, the weather code "95" corresponds to thunderstorms, so we can fill in the missing values for weather descriptions. 

This will be done in the database directly